<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/QFG_Specialist_Vs_Generalist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 8.3 MB/s eta 0:00:00


In [3]:

# ==============================================================================
#  PREAMBLE: IMPORTS AND SETUP
# ==============================================================================
import numpy as np
import time
import networkx as nx
from scipy.linalg import expm
import warnings

# Qiskit Imports
import qiskit
from qiskit.circuit import QuantumCircuit, ParameterVector, Gate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis.two_qubit import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate

# The powerful "Designer AI" engine
import cma

# Suppress benign warnings for cleaner output
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Qiskit version: {qiskit.__version__}")
print("\n--- QGF Experiment: Specialist vs. Generalist Bake-Off ---")

# ==============================================================================
#  CORE QGF ENGINE (UNCHANGED)
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_unitary_from_generator(coeffs: np.ndarray) -> np.ndarray:
    coeffs = np.asarray(coeffs).flatten()
    generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=coeffs)
    h_matrix = generator_h.to_matrix()
    return expm(-1j * h_matrix)

# ==============================================================================
#  PROBLEM DEFINITIONS: OUR BENCHMARK SUITE
# ==============================================================================

def get_max_cut_hamiltonian(n_qubits: int) -> SparsePauliOp:
    graph = nx.complete_graph(n_qubits)
    pauli_list = []
    for i, j in graph.edges():
        p_str = ['I'] * n_qubits
        p_str[i], p_str[j] = 'Z', 'Z'
        pauli_list.append("".join(p_str))
    return SparsePauliOp(pauli_list, coeffs=np.ones(len(pauli_list)))

def get_lih_hamiltonian(n_qubits: int) -> SparsePauliOp:
    # A simplified, pre-computed LiH Hamiltonian for 4 qubits.
    # In a real scenario, this would be generated by a chemistry library.
    if n_qubits != 4: raise ValueError("This LiH Hamiltonian is for 4 qubits.")
    paulis = [
        "IIXX", "IIXY", "IIYX", "IIYY", "IZIZ", "IZZI", "ZIIZ", "ZIZI",
        "XXII", "XYII", "YXII", "YYII", "ZZII"
    ]
    coeffs = [
        0.045, 0.045, -0.045, -0.045, 0.011, 0.011, 0.011, 0.011,
        0.045, -0.045, 0.045, -0.045, 0.011
    ]
    return SparsePauliOp(paulis, coeffs=coeffs)

def get_tfim_hamiltonian(n_qubits: int, h: float = 1.0) -> SparsePauliOp:
    # Transverse-Field Ising Model with periodic boundary conditions
    zz_terms, x_terms = [], []
    for i in range(n_qubits):
        # ZZ term
        p_zz = ['I'] * n_qubits
        p_zz[i], p_zz[(i + 1) % n_qubits] = 'Z', 'Z'
        zz_terms.append("".join(p_zz))
        # X term
        p_x = ['I'] * n_qubits
        p_x[i] = 'X'
        x_terms.append("".join(p_x))

    hamiltonian = SparsePauliOp(zz_terms, coeffs=-np.ones(n_qubits))
    hamiltonian += SparsePauliOp(x_terms, coeffs=-h * np.ones(n_qubits))
    return hamiltonian

def get_true_ground_state(hamiltonian: SparsePauliOp) -> np.ndarray:
    h_matrix = hamiltonian.to_matrix()
    eigvals, eigvecs = np.linalg.eigh(h_matrix)
    return eigvecs[:, 0]

# ==============================================================================
#  THE "FORGE AND TEST" PIPELINE
# ==============================================================================

def forge_and_test_specialist(problem_name: str, hamiltonian: SparsePauliOp):
    """The full pipeline: Forge a gate for a problem, then return the gate."""

    print(f"\n--- Forging Specialist Gate for: {problem_name} ---")
    n_qubits = hamiltonian.num_qubits
    true_ground_state = get_true_ground_state(hamiltonian)

    # --- ACT 1: THE FORGE ---
    print(f"  > [Act 1] Evolving gate generator...")

    # The fitness for the forge is now the final test drive fidelity!
    def forge_fitness(coeffs: np.ndarray) -> float:
        try:
            forged_u = build_unitary_from_generator(coeffs)

            # Synthesize into a usable gate
            gate = Gate(name="TempGate", num_qubits=2, params=[])
            decomposer = TwoQubitBasisDecomposer(CXGate())
            gate.definition = decomposer(forged_u)

            # Inner VQE to test this temporary gate
            def create_ansatz(gate_to_test):
                qc = QuantumCircuit(n_qubits)
                params = ParameterVector('θ', n_qubits * 2)
                for i in range(n_qubits): qc.ry(params[i], i)
                for i in range(n_qubits - 1): qc.append(gate_to_test, [i, i+1])
                for i in range(n_qubits): qc.ry(params[n_qubits + i], i)
                return qc

            ansatz = create_ansatz(gate)

            # We need a quick inner optimization to find the best Ry angles
            # A simpler, faster optimizer is fine here. Let's use a few random shots.
            best_fidelity = 0
            for _ in range(10): # 10 random shots
                ry_params = np.random.rand(ansatz.num_parameters) * 2 * np.pi
                bound_circ = ansatz.assign_parameters(ry_params)
                state_vec = Statevector.from_instruction(bound_circ).data
                fidelity = qiskit.quantum_info.state_fidelity(true_ground_state, state_vec)
                if fidelity > best_fidelity:
                    best_fidelity = fidelity

            return 1.0 - best_fidelity
        except Exception:
            return 1.0

    # CMA-ES for the forge
    x0_forge = np.random.rand(15) * 2 * np.pi - np.pi
    sigma0_forge = 0.5
    options_forge = {'bounds': [-np.pi, np.pi], 'maxfevals': 1000, 'verbose': -9}
    es_forge = cma.CMAEvolutionStrategy(x0_forge, sigma0_forge, options_forge)
    es_forge.optimize(forge_fitness)

    best_coeffs = es_forge.result.xbest
    final_fitness = es_forge.result.fbest
    print(f"  > Forge complete. Best test fidelity found: {1.0 - final_fitness:.2%}")

    # --- Synthesize the final, best gate for this specialist ---
    final_forged_u = build_unitary_from_generator(best_coeffs)
    final_gate = Gate(name=f"{problem_name}_Specialist", num_qubits=2, params=[])
    decomposer = TwoQubitBasisDecomposer(CXGate())
    final_gate.definition = decomposer(final_forged_u)

    return final_gate

def run_final_evaluation(gate_to_test: Gate, problem_name: str, hamiltonian: SparsePauliOp) -> float:
    """The final bake-off test drive."""
    n_qubits = hamiltonian.num_qubits
    true_ground_state = get_true_ground_state(hamiltonian)

    def create_ansatz(gate):
        qc = QuantumCircuit(n_qubits)
        params = ParameterVector('θ', n_qubits * 2)
        for i in range(n_qubits): qc.ry(params[i], i)
        for i in range(n_qubits - 1): qc.append(gate, [i, i+1])
        for i in range(n_qubits): qc.ry(params[n_qubits + i], i)
        return qc

    ansatz = create_ansatz(gate_to_test)

    def fitness(ry_params):
        state_vec = Statevector.from_instruction(ansatz.assign_parameters(ry_params)).data
        return 1.0 - qiskit.quantum_info.state_fidelity(true_ground_state, state_vec)

    x0 = np.random.rand(ansatz.num_parameters) * 2 * np.pi
    sigma0 = 0.5
    options = {'bounds': [0, 2*np.pi], 'maxfevals': 500, 'verbose': -9}
    es = cma.CMAEvolutionStrategy(x0, sigma0, options)
    es.optimize(fitness)

    return 1.0 - es.result.fbest


# ==============================================================================
#  MAIN SCRIPT
# ==============================================================================

if __name__ == "__main__":

    N_QUBITS = 4 # Using 4 qubits for all problems for consistency

    # 1. Define our problem suite
    problems = {
        "Max-Cut": get_max_cut_hamiltonian(N_QUBITS),
        "LiH_Chem": get_lih_hamiltonian(N_QUBITS),
        "TFIM_Phys": get_tfim_hamiltonian(N_QUBITS)
    }

    # 2. Forge a specialist gate for each problem
    specialist_gates = {}
    for name, H in problems.items():
        specialist_gates[name] = forge_and_test_specialist(name, H)

    # 3. The Final Bake-Off
    print(f"\n\n{'='*70}\n--- FINAL BAKE-OFF RESULTS ---")
    print(f"{'='*70}\n")

    # Prepare header
    header = f"{'↓ Tested On / Forged For →':<28}"
    for name in specialist_gates.keys():
        header += f"| {name:<15}"
    print(header)
    print("-" * len(header))

    # Run tests and print rows
    results_matrix = {}
    for prob_name, H in problems.items():
        row_str = f"{prob_name:<28}"
        results_matrix[prob_name] = {}
        for spec_name, gate in specialist_gates.items():
            print(f"Testing {spec_name} gate on {prob_name} problem...")
            fidelity = run_final_evaluation(gate, prob_name, H)
            results_matrix[prob_name][spec_name] = fidelity
            row_str += f"| {fidelity:<15.2%}"
        print(row_str)

    print("-" * len(header))

    # 4. Analyze the results
    print("\n--- Analysis ---")
    # Check for a generalist
    avg_performance = {name: np.mean([results_matrix[p][name] for p in problems]) for name in specialist_gates}
    best_generalist = max(avg_performance, key=avg_performance.get)

    # Check for specialists
    is_specialist_dominant = True
    for name in problems:
        if max(results_matrix[name], key=results_matrix[name].get) != name:
            is_specialist_dominant = False
            break

    if is_specialist_dominant:
        print("Conclusion: Specialist gates dominate! The AI discovered that the optimal gate")
        print("is highly dependent on the problem's structure. This points towards a future")
        print("of co-designing hardware and algorithms for specific problem classes.")
    else:
        print(f"Conclusion: A Generalist has emerged! The '{best_generalist}' gate performed best")
        print("on average, suggesting it's a powerful, new, all-purpose quantum gate.")

Qiskit version: 2.1.1

--- QGF Experiment: Specialist vs. Generalist Bake-Off ---

--- Forging Specialist Gate for: Max-Cut ---
  > [Act 1] Evolving gate generator...
  > Forge complete. Best test fidelity found: 58.75%

--- Forging Specialist Gate for: LiH_Chem ---
  > [Act 1] Evolving gate generator...
  > Forge complete. Best test fidelity found: 51.25%

--- Forging Specialist Gate for: TFIM_Phys ---
  > [Act 1] Evolving gate generator...
  > Forge complete. Best test fidelity found: 59.97%


--- FINAL BAKE-OFF RESULTS ---

↓ Tested On / Forged For →  | Max-Cut        | LiH_Chem       | TFIM_Phys      
-------------------------------------------------------------------------------
Testing Max-Cut gate on Max-Cut problem...
Testing LiH_Chem gate on Max-Cut problem...
Testing TFIM_Phys gate on Max-Cut problem...
Max-Cut                     | 93.97%         | 46.48%         | 75.23%         
Testing Max-Cut gate on LiH_Chem problem...
Testing LiH_Chem gate on LiH_Chem problem...
Testin